# Importing Dependencies

In [38]:
# System Packages

import os
from pathlib import Path
from dotenv import load_dotenv
from typing import TypedDict, Annotated

# Langchain packages

from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter

# LangGraph packages

from langgraph.graph import START, END, StateGraph
from langgraph.graph.message import add_messages

# LLM setup

In [40]:
load_dotenv()

# Chat Model LLM API key
llm_api_key = os.getenv("OPENROUTER_API_KEY")

if llm_api_key:
    print(f"----------\nLLM API key fetch success! \n API = {llm_api_key[:10]}...\n")
else:
    print(f"----------\nAPI KEY = NONE\n")

# Embedding Model API Key
embeed_api_key = os.getenv("OPENROUTER_EMBEDDING_API")

if embeed_api_key:
    print(f"----------\nEmbedding API key fetch success! \n API = {embeed_api_key[:10]}...\n")
else:
    print(f"----------\nEmbedding API KEY = NONE\n")


----------
LLM API key fetch success! 
 API = sk-or-v1-1...

----------
Embedding API key fetch success! 
 API = sk-or-v1-6...



In [29]:
llm = ChatOpenAI(
    base_url = "https://openrouter.ai/api/v1",
    model = "nvidia/nemotron-3.5-lightning:free",
    api_key = llm_api_key,
    temperature = 0.5
)

try:
    llm_init_status = llm.invoke("Are you available? status 200?")
    if llm_init_status:
        print(f"----------\nStatus : 200 \nLLM responding : {llm_init_status.content}\n")
    else:
        print("----------\nStatus : 400 \nLLM un-responsive\n")
except esceptions as e:
    print(f"Error : {e}")

----------
Status : 200 
LLM responding : Status: 200 OK. I'm online and ready to help! 😊 How can I assist you today?



# RAG

Setting PDF path

In [30]:
#Only for .ipynb file
root_path = Path.cwd().parent

#Only for .py file
# root_path = Path(__file__).resolve().parent.parent

print(f"----------\nRoot path = {root_path}\n")


# Path to pdf
pdf_path = root_path / "resource" / "muse.pdf"
print(f"----------\nPDF path = {pdf_path} \n Status: {pdf_path.exists()}\n")

----------
Root path = /home/program/Documents/KEC/MajorProject/Cudo

----------
PDF path = /home/program/Documents/KEC/MajorProject/Cudo/resource/muse.pdf 
 Status: True



Loading Docs

In [31]:
# Loader

loader = PyPDFLoader(pdf_path)
docs = loader.load()
print(f"----------\nTotal Page Count = {len(docs)} \n")
print(f"----------\nAttributes of Document = {list(vars(docs[0]).keys())}\n")
print(f"----------\n---------- Page 1 ---------- \n {docs[0]}\n")

----------
Total Page Count = 10 

----------
Attributes of Document = ['id', 'metadata', 'page_content', 'type']

----------
---------- Page 1 ---------- 
 page_content='AI AGENTS LIBRARY BEGINNER GUIDE / 2026
FREE PDF GUIDE
How to Use
Meta Muse
Beginner Tutorial + 10 Things to Try
Set permissions, connect apps, give Muse its first tasks, and try 10 practical workflows - from reminders
and research to dashboards, forms, and goal planning.
Last verified: September 22, 2026
Source: Meta Muse Help Center + official Muse page
Note: Research-based guide; AI Agents Library has not completed a hands-on Muse test yet.
Get Your Free AI Roadmap
aiagentslibrary.com' metadata={'producer': 'GPL Ghostscript 9.55.0', 'creator': 'AI Agents Library', 'creationdate': "D:20260922130702Z00'00'", 'moddate': "D:20260922130702Z00'00'", 'author': 'AI Agents Library', 'title': 'How to Use Meta Muse: Beginner Tutorial + 10 Things to Try', 'subject': 'Beginner guide to using Meta Muse', 'keywords': '', 'source'

Splitting Documents

In [37]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 800,
    chunk_overlap = 100
)


chunks = splitter.split_documents(docs)

print(f"----------\nLength of chunks = {len(chunks)}\n")
print(f"----------\nChunks = \n{chunks}\n")
print(f"----------\nChunk 0 content =\n{chunks[0].page_content}\n")
print(f"----------\nChunk 1 content =\n{chunks[1].page_content}\n")

----------
Length of chunks = 27

----------
Chunks = 
[Document(metadata={'producer': 'GPL Ghostscript 9.55.0', 'creator': 'AI Agents Library', 'creationdate': "D:20260922130702Z00'00'", 'moddate': "D:20260922130702Z00'00'", 'author': 'AI Agents Library', 'title': 'How to Use Meta Muse: Beginner Tutorial + 10 Things to Try', 'subject': 'Beginner guide to using Meta Muse', 'keywords': '', 'source': '/home/program/Documents/KEC/MajorProject/Cudo/resource/muse.pdf', 'total_pages': 10, 'page': 0, 'page_label': '1'}, page_content='AI AGENTS LIBRARY BEGINNER GUIDE / 2026\nFREE PDF GUIDE\nHow to Use\nMeta Muse\nBeginner Tutorial + 10 Things to Try\nSet permissions, connect apps, give Muse its first tasks, and try 10 practical workflows - from reminders\nand research to dashboards, forms, and goal planning.\nLast verified: September 22, 2026\nSource: Meta Muse Help Center + official Muse page\nNote: Research-based guide; AI Agents Library has not completed a hands-on Muse test yet.\nGet Your 

Create Embeddings for Chunks